In [ ]:
import numpy as np
import pandas as pd

orders = pd.read_csv("data/orders.csv")
products = pd.read_csv("data/products.csv")
order_products = pd.read_csv("data/order_products__prior.csv")

user_ids_sample = orders["user_id"].drop_duplicates().sample(n=1000, replace=False).reset_index()["user_id"]


0        6207
1       64976
2      131884
3       71572
4      168484
        ...  
995      5342
996     48723
997    116791
998     48570
999    202600
Name: user_id, Length: 1000, dtype: int64

In [ ]:
df = (order_products
    .merge(orders[["order_id", "user_id"]], how='left', on='order_id')
    .merge(user_ids_sample, how='inner', on='user_id')
)



In [20]:

df_pivoted = (df
              .pivot_table(index="order_id", columns="product_id",aggfunc='size', fill_value=0)
              .merge(orders[["order_id", "user_id"]], how='left', on='order_id'))


# Median doesnt work, will always be zero:

In [21]:

df_pivoted.median(axis=0)


order_id    1696538.0
10                0.0
18                0.0
23                0.0
25                0.0
              ...    
49679             0.0
49680             0.0
49681             0.0
49683             0.0
user_id      104421.0
Length: 15094, dtype: float64

In [22]:

mean_products_pr_order = df_pivoted.mean(axis=0).rename('product_average').to_frame().iloc[1:-1]


In [25]:
n_user_orders = orders.query("eval_set == 'prior'").groupby("user_id").size().reset_index(name='n_orders')

(df[["user_id", "product_id"]]
 .groupby(["user_id", "product_id"])
 .size()
 .reset_index(name='n_products')
 .merge(n_user_orders, on='user_id', how='left')
 .assign(user_product_average = lambda row: row["n_products"] / row["n_orders"])
 .merge(mean_products_pr_order, left_on=["product_id"], right_index=True, how='left')
 .assign(popularity_metric = lambda row: row["user_product_average"] - row["product_average"])
)



,user_id,product_id,n_products,n_orders,user_product_average,product_average,popularity_metric
0,25,278,1,3,0.333333,0.000462,0.332871
1,25,1216,1,3,0.333333,0.000198,0.333135
2,25,10096,1,3,0.333333,0.000066,0.333267
3,25,11734,1,3,0.333333,0.000066,0.333267
4,25,21506,1,3,0.333333,0.001057,0.332276
...,...,...,...,...,...,...,...
63038,206014,48205,2,3,0.666667,0.005417,0.661250
63039,206014,49683,2,3,0.666667,0.029132,0.637535
63040,206100,9434,3,3,1.000000,0.001784,0.998216
63041,206100,27156,1,3,0.333333,0.013872,0.319461
